# Exercice 02 – Un mini-chat en ligne de commande

## Objectif

Construire une boucle de conversation interactive dans le notebook : l'utilisateur tape un message, le modèle répond **en streaming**, et la conversation garde sa **mémoire** d'un tour à l'autre.

## Ce que vous allez pratiquer

- Le streaming (`stream=True`, `delta`, `flush`)
- La gestion de l'historique côté client (rôles `user` / `assistant`)
- Les champs `usage` et `timings`
- La limite de contexte et une stratégie pour ne pas la dépasser

## Prérequis

Un `llama-server` lancé avec `-c 8192 -np 1` (comme dans le cours).

## Fonctionnalités à implémenter

| # | Fonctionnalité | Obligatoire |
|---|---|---|
| 1 | Boucle `while True` avec `input()`, sortie avec `quit` | oui |
| 2 | Réponse affichée en **streaming** | oui |
| 3 | Historique correct : le modèle se souvient des tours précédents | oui |
| 4 | Commande `/reset` : vide l'historique en gardant le system prompt | oui |
| 5 | Commande `/history` : affiche le nombre de messages et de tokens du dernier prompt | oui |
| 6 | Après chaque réponse : afficher `tokens générés` et `tok/s` | bonus |
| 7 | Alerte quand le prompt dépasse 80 % du contexte | bonus |
| 8 | **Fenêtre glissante** : supprimer les plus vieux échanges quand on approche de la limite | bonus |
| 9 | Commande `/save fichier.json` : sauvegarder l'historique | bonus |

## Critères de réussite

- Taper « Je m'appelle Léa et j'habite à Namur. » puis « Où est-ce que j'habite ? » → le modèle répond Namur.
- `/reset` puis « Où est-ce que j'habite ? » → le modèle ne sait plus.
- Le texte apparaît progressivement, pas d'un bloc.
- Aucun `None` n'apparaît dans l'affichage.

In [ ]:
import json
import time

import requests
from openai import OpenAI

In [ ]:
BASE_URL = "http://127.0.0.1:8080"
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="pas-besoin")

SYSTEM_PROMPT = "Tu es un assistant concis et sympathique. Tu réponds en français."

print(requests.get(f"{BASE_URL}/health").json())

## Étape 1 – Connaître la taille du contexte

Plutôt que d'écrire `8192` en dur, demandez-la au serveur. L'endpoint `/props` renvoie un dictionnaire ; la clé `default_generation_settings` contient `n_ctx`, la taille de contexte **par slot**.

Stockez-la dans une variable `N_CTX`.

In [ ]:
# TODO : récupérer N_CTX depuis /props
N_CTX = ...
print("Contexte par slot :", N_CTX)

## Étape 2 – La fonction de réponse en streaming

Écrivez `repondre(history: list) -> str` qui :

1. envoie `history` au modèle avec `stream=True`,
2. affiche chaque morceau au fur et à mesure,
3. renvoie le texte complet assemblé.

> **Tips**
> - `print(delta, end="", flush=True)` sinon rien ne s'affiche avant la fin.
> - Testez `if chunk.choices and chunk.choices[0].delta.content:` avant d'afficher : le premier et le dernier chunk ont un `content` à `None`.
> - Pour les stats (bonus 6) : ajoutez `stream_options={"include_usage": True}` à l'appel. Le dernier chunk porte alors `chunk.usage` et, côté llama-server, `chunk.model_extra.get("timings")`. Si `timings` est absent dans votre version, mesurez le temps avec `time.perf_counter()` et divisez `usage.completion_tokens` par la durée.
> - Sur Qwen3, le raisonnement arrive dans `delta.reasoning_content` : soit vous l'ignorez, soit vous désactivez le mode think via `extra_body={"chat_template_kwargs": {"enable_thinking": False}}`.

In [ ]:
def repondre(history: list) -> str:
    # TODO
    ...

# Test rapide
_ = repondre([{"role": "user", "content": "Dis bonjour en une phrase."}])

## Étape 3 – La boucle de chat

Écrivez `chat()` :

```
history = [system prompt]
tant que vrai :
    lire la saisie
    si "quit"      → sortir
    si "/reset"    → réinitialiser history
    si "/history"  → afficher des infos
    sinon :
        ajouter le message user à history
        appeler repondre(history)
        ajouter la réponse dans history avec le bon rôle
```

> **Tips**
> - Quel rôle pour la réponse du modèle dans l'historique ? (Relisez la partie du cours sur `assistant` vs `system`.)
> - `input()` bloque la cellule : c'est normal. Tapez `quit` pour la libérer. Si vous interrompez le kernel, pensez à ré-exécuter le setup.
> - Gardez `history` en variable globale ou renvoyez-le à la fin de `chat()` : c'est pratique pour l'inspecter après coup avec `pprint`.

In [ ]:
def chat():
    history = [{"role": "system", "content": SYSTEM_PROMPT}]
    # TODO
    return history

history = chat()

## Étape 4 (bonus) – Alerte et fenêtre glissante

Le nombre de tokens du prompt est dans `usage.prompt_tokens` (ou dans le dernier chunk si vous streamez avec `include_usage`).

1. Après chaque réponse, si `prompt_tokens > 0.8 * N_CTX`, affichez un avertissement.
2. Fenêtre glissante : tant que le prompt dépasse le seuil, supprimez le plus vieux **couple** user/assistant (positions 1 et 2 de la liste, le system prompt reste en 0).

> **Tips**
> - Pour tester sans attendre 50 tours, lancez temporairement le serveur avec `-c 1024`, ou collez un long texte dans le chat.
> - Vérifiez avec Léa/Namur : au bout de quelques tours après suppression, le modèle doit avoir oublié.

In [ ]:
# TODO : version améliorée de chat()

## Questions de réflexion

1. Pourquoi la réponse du modèle est-elle stockée avec le rôle `assistant` et pas `system` ?
2. Que se passe-t-il si vous oubliez d'ajouter la réponse du modèle à `history` ? Testez.
3. Le tour 10 n'est pas 10 fois plus lent que le tour 1 alors qu'on renvoie tout l'historique. Pourquoi ? (Indice : `timings["prompt_n"]` vs `usage.prompt_tokens`.)
4. Quels sont les inconvénients de la fenêtre glissante ? Proposez une autre stratégie pour gérer une très longue conversation.

*Vos réponses ici…*